# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Freestyle submission.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**The principle it serves:** attack your own model before anyone else does. A score that looks too good is
investigated, not celebrated. This notebook builds the feature vector, documents each feature, and records the three
attacks that changed the study.

In [1]:
# Setup: works locally and in Colab. Installs only missing packages and never prints a token.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "huggingface_hub", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
DATA = wh.data_available(try_download=True)   # needs HF_TOKEN as an env var or Colab secret
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8])
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

pinned warehouse snapshot available: True | revision 50cbf7c3


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Question:** what exactly goes into the model? **Why it matters (FlyRank principle: every feature must exist before the
moment I predict):** the vector is the only thing the model sees. **Method:** one function builds 34 features per page
and origin from months *o-3* to *o-1*; the list is exported from code so this notebook, the paper and the model share it.

In [2]:
feats = receipt("erosion_features.json")
print(f"{len(feats)} features exported by the model code")
if DATA:
    from erosion_model import load, make_features
    df, grid, vals = load(); X = make_features(df, grid, vals)
    print("feature matrix:", X.shape, "| columns match the exported list:", set(X.columns) == set(feats))

34 features exported by the model code


feature matrix: (68662, 34) | columns match the exported list: True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Question:** for each feature, what does it mean, how are missing values handled, which are categorical, and does it
exist before the decision? **Why it matters (FlyRank principle: null is not zero):** a silent fill can fake a signal.

- **Available when:** every feature is computed from months *o-3* to *o-1*, or from labels whose outcome window closed
  by *o-1* (`client_prev_erode_rate`, `page_prev_ratio`). Nothing is read from month *o* or later.
- **Missing values.** Missingness is structural (no GA4 for the client, no matured history yet). The logistic model
  fills with the median and adds a missing-indicator column; gradient boosting handles missing values natively.
  `l_org_sess` is NaN, not zero, when no GA4 day exists, and `has_ga4` carries that fact.
- **Categorical.** Main intent is one-hot (`intent_informational`, `intent_transactional`, `intent_commercial`,
  `intent_navigational`), and content type is a flag (`is_feedly`).

In [3]:
fam = {"l_imp_f": "exposure", "l_clk_f": "exposure", "imp_growth": "trajectory", "ctr_growth": "trajectory", "imp_growth_prev": "trajectory",
       "imp_l14_share": "trajectory", "ctr_recency": "trajectory", "click_day_share": "stability", "spikiness": "stability",
       "pos_f": "rank", "pos_delta": "rank", "pos_std": "rank", "ctr_vs_ref": "CTR vs expectation", "ctr_vs_client": "CTR vs expectation"}
t = pd.DataFrame([{"feature": k, "family": fam.get(k, "client context" if k.startswith(("c_", "client")) else "page history" if k.startswith("page_")
                   else "channels (GA4)" if k in ("has_ga4", "l_org_sess", "has_ai_referral") else "content"), "meaning": v} for k, v in feats.items()])
display(t)

,feature,family,meaning
0,l_imp_f,exposure,"log impressions, feature window (2 months)"
1,l_clk_f,exposure,"log clicks, feature window"
2,imp_growth,trajectory,log impressions month -1 vs month -2
3,ctr_growth,trajectory,log CTR month -1 vs month -2 (clicks growth mi...
4,imp_growth_prev,trajectory,log impressions month -2 vs month -3
5,imp_l14_share,trajectory,share of month -1 impressions in its last 14 days
6,ctr_recency,trajectory,log CTR last 14 days vs whole feature window
7,click_day_share,stability,days with a click / days with an impression
8,spikiness,stability,max single-day impressions / window impressions
9,pos_f,rank,"impression-weighted position, feature window"


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Question:** can the label be predicted from momentum that is already visible, and does my harness see a planted leak?
**Why it matters:** that failure ended my Lane 2 study (a two-line momentum rule beat every model). **Method:** three
label versions, each run only on the three development months. The sealed month stayed closed.

**Attack 1: the label leaked twice, and the models looked better while it did.**

In [4]:
log = receipt("label_design_log.json")
rows = []
for v, d in log["versions"].items():
    if "p50" in d:
        for m, vals in d["p50"].items():
            rows.append({"version": v, "ranker": m, **dict(zip(["2026-02", "2026-03", "2026-04"], vals))})
dev = receipt("erosion_model_metrics.json")["origins"]
for m in ["rule_position_slipping", "logreg", "hgb"]:
    rows.append({"version": "v3_final", "ranker": m, **{o: dev[o]["metrics"][m]["p@50"] for o in dev}})
display(pd.DataFrame(rows))
for v, d in log["versions"].items():
    print(f"{v}: {d.get('problem_found', 'final version')}")

,version,ranker,2026-02,2026-03,2026-04
0,v1_anchor_two_month_average,rule_position_slipping,0.46,0.76,0.50
1,v1_anchor_two_month_average,logreg,0.68,0.60,0.74
2,v1_anchor_two_month_average,hgb,0.90,0.72,0.88
3,v2_anchor_last_month_shrunk_to_own_history,rule_position_slipping,0.48,0.82,0.56
4,v2_anchor_last_month_shrunk_to_own_history,logreg,0.64,0.68,0.72
5,v2_anchor_last_month_shrunk_to_own_history,hgb,0.80,0.68,0.80
6,v3_final,rule_position_slipping,0.46,0.76,0.50
7,v3_final,logreg,0.52,0.60,0.66
8,v3_final,hgb,0.44,0.74,0.74


v1_anchor_two_month_average: a page whose CTR already fell in month -1 counts as 'eroding' even if it only stays at its new level: momentum leaks into the label
v2_anchor_last_month_shrunk_to_own_history: for a declining page its own history is higher than its present, so shrinking toward it still inflates the expectation (the same leak through the back door)
v3_final_anchor_last_month_shrunk_to_client_prior: final version


**What it means.** The leaky versions made the learned models look much better (gradient boosting reached 0.90 at 50
pages). After the fix, every model score dropped. That drop is the evidence that the leak was real, and the honest
lower numbers are the ones this submission claims.

**Attack 2: does the harness see a planted leak?** **Method:** add a noisy copy of the outcome ratio as a feature on the
April forward split. If precision does not jump toward 1.0, the harness itself is broken.

In [5]:
lk = receipt("erosion_leakage_checks.json")
display(pd.DataFrame({k: v for k, v in lk.items() if k.startswith("forward")}).T)

,p@50,ap
forward_2026-04_honest,0.66,0.393
forward_2026-04_planted_future_ratio,1.00,0.769


**What it means.** Precision@50 goes from 0.66 to 1.00 with the planted feature, so the evaluation can see a leak. The
honest model has no such feature.

**Attack 3: does the split decide the answer?** **Question:** how much does a random row split flatter the model?
**Why it matters:** pages from one client share hidden character, so a random split lets the model recognise the client
instead of learning erosion.

In [6]:
split = pd.DataFrame({k: lk[k] for k in ["random_row_kfold", "client_group_kfold"]}).T
display(split)
display(pd.DataFrame(lk["unseen_client_within_origin"]).T.assign(lift=lambda d: (d.ap_mean / d.base_mean).round(2)))

,ap_mean,ap_std,base_rate
random_row_kfold,0.5383,0.0150,0.2922
client_group_kfold,0.4223,0.0712,0.2922


,ap_mean,base_mean,lift
2025-12,0.4410,0.3274,1.35
2026-01,0.3299,0.2467,1.34
2026-02,0.3622,0.2388,1.52
2026-03,0.4913,0.3249,1.51
2026-04,0.4225,0.3064,1.38


**What it means.** A random row split reports PR-AUC 0.538. Holding whole clients out gives 0.422. The 0.116 gap is
client memorisation, so every claim in this submission uses forward months and reports held-out-client checks. Within
every month, the model still ranks pages of clients it never saw above the base rate (lift column).

**Attack checklist.**

| Check | Status | Where |
|---|---|---|
| Timeline drawn: all features strictly before the label window | done | ML-03, `erosion_frame.py` |
| No label-derived or sibling columns in the features | done; planted-leak test above | ML-04, attack 2 here |
| No product flags or existing-system scores as features | done; the warehouse release has none, `provider_used` excluded | ML-04 |
| Population selection checked for outcome-window information | disclosed: measurability uses outcome impressions | ML-04 |
| Split grouped by the repeating entity and by time | forward origins plus held-out clients | ML-08, ML-09 |
| Base rate printed next to every metric | done | all receipts |
| Top feature importance sanity-checked | done; the top driver is explained, not celebrated | ML-08 |
| Metrics out of sample | forward only; tuning inside training months | ML-08 |
| Sealed holdout leaves receipts | `erosion_freeze.json`, `erosion_model_metrics_sealed.json` | ML-09 |

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Question:** which fields did I refuse to use, and why? **Why it matters (FlyRank principle: privacy and leakage are
decided before modelling):** an exclusion with a reason can be audited.

| Excluded field | Why |
|---|---|
| `clk_out`, `imp_out`, `pos_out`, `exp_out` | outcome-window columns, the label's own ingredients (attack 2 shows what they do) |
| `provider_used` | contains strings that look like client brand names; never selected |
| the 90-day query table | covers April to June 2026, which overlaps every outcome window |
| product flags and existing-system scores | the warehouse release has none |
| `model_used`, `last_optimized_date` | not a valid treatment (latest regeneration, not author) and too little follow-up time (8.7% of pages, all after 24 April 2026) |

In [7]:
banned = ["imp_out", "clk_out", "pos_out", "exp_out", "erode", "provider_used", "model_used", "last_optimized_date"]
print("excluded fields present in the 34-feature list (must be empty):", [c for c in banned if c in feats])

excluded fields present in the 34-feature list (must be empty): []


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.